
# UNet Fine-Tuning for Lake Segmentation — **RGB from Planet 8-Band**

This notebook trains a **UNet** on **RGB** channels extracted from **Planet 8-band** GeoTIFF tiles (size **512×512**).  
We use **ImageNet-pretrained** encoders (ResNet, EfficientNet, MobileNet, …) so transfer learning works as intended.

**What it does**
- Randomly sample **1,000** image/label pairs by matching filenames.
- Extract **RGB** from Planet 8-band tiles (defaults to **(6,4,2) = (Red, Green, Blue)** for SuperDove SR).
- Train/Val split (80/20).
- Train UNet across multiple **backbones** using **pretrained(ImageNet)** weights.
- **Loss:** BCE + Soft Dice (good for class imbalance).
- **Metrics:** IoU, Precision, Recall, F1, Accuracy on **train + val**, logged to CSV.
- Save **best-train-IoU** checkpoint per backbone.

> If your band ordering is different, **change `RGB_BANDS`** below.


## Dependencies

In [ ]:

# If needed, uncomment to install:
# %pip install -q torch torchvision timm segmentation-models-pytorch rasterio albumentations pandas scikit-image


## Configuration

In [10]:

from pathlib import Path
import random

# === REQUIRED: Paths ===
IMAGES_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/images")  # change to your imagery folder
LABELS_DIR = Path("D:/Thesis/glacial_lake_detection_thesis/Training/3 Preparing train data/rgb_outputs/subset_750/labels")  # change to your label folder (same filenames)

# === PlanetScope SuperDove SR default band mapping (1-indexed) ===
RGB_BANDS = (3, 2, 1)  # (R, G, B) 1-indexed

# === Outputs ===
OUTPUT_DIR = Path("./training_output_rgb/subset_750")
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# === Training config ===
NUM_SAMPLES = 750
RANDOM_SEED = 42
VAL_RATIO = 0.2
BATCH_SIZE = 2
NUM_WORKERS = 0
EPOCHS = 10
LR = 1e-3

BACKBONES = ['mit_b0', 'mit_b1', 'mit_b2', 'mit_b3', 'mit_b4', 'mit_b5']

random.seed(RANDOM_SEED)


## Discover Pairs & Sample

In [2]:

def list_pairs(images_dir: Path, labels_dir: Path):
    img_files = {p.name: p for p in images_dir.glob("*.tif")}
    lbl_files = {p.name: p for p in labels_dir.glob("*.tif")}
    common = sorted(set(img_files) & set(lbl_files))
    return [(img_files[n], lbl_files[n]) for n in common]

all_pairs = list_pairs(IMAGES_DIR, LABELS_DIR)
if not all_pairs:
    raise RuntimeError(f"No paired .tif files found under {IMAGES_DIR} & {LABELS_DIR}.")

print(f"Found {len(all_pairs)} pairs.")
random.shuffle(all_pairs)
sampled_pairs = all_pairs[:min(NUM_SAMPLES, len(all_pairs))]

split_idx = int(len(sampled_pairs) * (1 - VAL_RATIO))
train_pairs = sampled_pairs[:split_idx]
val_pairs = sampled_pairs[split_idx:]
print(f"Train: {len(train_pairs)}, Val: {len(val_pairs)}")


Found 750 pairs.
Train: 600, Val: 150



## Dataset & Dataloaders (RGB extraction)
- Reads 8-band imagery and **selects RGB channels** via `RGB_BANDS` (1-indexed).
- Per-image **min–max** normalization to [0,1].
- Light flips for augmentation.


In [3]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
import rasterio
import albumentations as A

def read_geotiff(path: Path):
    with rasterio.open(path) as src:
        arr = src.read()  # (C,H,W), 1-indexed bands in file
    return arr

def select_rgb(x: np.ndarray, rgb_bands=(6,4,2)):
    idx = [b-1 for b in rgb_bands]
    return x[idx, ...].astype(np.float32)

def minmax_per_band(x: np.ndarray, eps=1e-6):
    x = x.astype(np.float32)
    for i in range(x.shape[0]):
        vmin = np.min(x[i]); vmax = np.max(x[i])
        if vmax - vmin < eps:
            x[i] = 0.0
        else:
            x[i] = (x[i] - vmin) / (vmax - vmin + eps)
    return x

class LakeTilesRGB(Dataset):
    def __init__(self, pairs, rgb_bands=(6,4,2), aug=None, normalize=True):
        self.pairs = pairs
        self.rgb_bands = rgb_bands
        self.aug = aug
        self.normalize = normalize

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        img8 = read_geotiff(img_path)          # (8,H,W)
        img = select_rgb(img8, self.rgb_bands) # (3,H,W)

        with rasterio.open(lbl_path) as src:
            lbl = src.read(1)                  # (H,W)
        lbl = (lbl > 0).astype(np.float32)

        if self.normalize:
            img = minmax_per_band(img)

        # Albumentations expects HWC
        img_hwc = np.transpose(img, (1,2,0))
        lbl_hwc = lbl[..., None]

        if self.aug:
            transformed = self.aug(image=img_hwc, mask=lbl_hwc)
            img_hwc = transformed["image"]
            lbl_hwc = transformed["mask"]

        # Back to CHW
        img = torch.from_numpy(np.transpose(img_hwc, (2,0,1))).float()
        lbl = torch.from_numpy(lbl_hwc[...,0]).float()
        return img, lbl

train_aug = A.Compose([A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5)])
val_aug   = A.Compose([])

train_ds = LakeTilesRGB(train_pairs, rgb_bands=RGB_BANDS, aug=None, normalize=True)
val_ds   = LakeTilesRGB(val_pairs,   rgb_bands=RGB_BANDS, aug=None,   normalize=True)

PIN_MEMORY = False
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          pin_memory=PIN_MEMORY)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          pin_memory=PIN_MEMORY)

len(train_ds), len(val_ds)


(600, 150)

## Loss & Metrics

In [4]:

import torch.nn.functional as F

def dice_loss_with_logits(logits, targets, eps=1e-7):
    probs = torch.sigmoid(logits)
    if probs.dim() == 4:
        probs = probs[:,0]
    intersection = (probs * targets).sum(dim=(1,2))
    union = probs.sum(dim=(1,2)) + targets.sum(dim=(1,2)) + eps
    dice = (2 * intersection + eps) / union
    return 1 - dice.mean()

def bce_dice_loss(logits, targets):
    return F.binary_cross_entropy_with_logits(logits, targets) + dice_loss_with_logits(logits, targets)

def compute_metrics_from_logits(logits, targets):
    with torch.no_grad():
        probs = torch.sigmoid(logits)
        preds = (probs > 0.5).float()
        t = targets.float()
        tp = (preds * t).sum(dim=(1,2))
        tn = ((1-preds) * (1-t)).sum(dim=(1,2))
        fp = (preds * (1-t)).sum(dim=(1,2))
        fn = ((1-preds) * t).sum(dim=(1,2))
        eps = 1e-7
        precision = (tp / (tp + fp + eps)).mean().item()
        recall    = (tp / (tp + fn + eps)).mean().item()
        f1        = (2*precision*recall / (precision + recall + eps))
        iou       = (tp / (tp + fp + fn + eps)).mean().item()
        accuracy  = ((tp + tn) / (tp + tn + fp + fn + eps)).mean().item()
        return dict(iou=iou, precision=precision, recall=recall, f1=f1, accuracy=accuracy)


## Model & Training (ImageNet-pretrained encoders, in_channels=3)

In [8]:

import segmentation_models_pytorch as smp
import torch
from torch import optim
import pandas as pd
from datetime import datetime
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

def make_segformer_rgb(backbone: str, num_classes: int = 1, pretrained=True):
    encoder_weights = "imagenet" if pretrained else None
    model = smp.Unet(
        encoder_name=backbone,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=num_classes,
    )
    return model

def run_one_epoch(model, loader, optimizer=None, use_amp=True):
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = 0.0; n_batches = 0
    agg = dict(iou=0.0, precision=0.0, recall=0.0, f1=0.0, accuracy=0.0)

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        if is_train: optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(imgs)[:,0]
            loss = bce_dice_loss(logits, lbls)

        if is_train:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item(); n_batches += 1
        m = compute_metrics_from_logits(logits.detach(), lbls.detach())
        for k in agg: agg[k] += m[k]

    avg_loss = total_loss / max(1, n_batches)
    for k in agg: agg[k] /= max(1, n_batches)
    torch.cuda.empty_cache()
    return avg_loss, agg

# Determinism tweaks (optional)
#torch.backends.cudnn.benchmark = False
#torch.backends.cudnn.deterministic = True


Device: cuda


In [ ]:
results_csv = OUTPUT_DIR / "logs" / "results.csv"
if not results_csv.exists():
    pd.DataFrame(columns=[
        "timestamp","backbone","epoch","split",
        "loss","iou","precision","recall","f1","accuracy"
    ]).to_csv(results_csv, index=False)

for backbone in BACKBONES:
    print(f"\n=== Training backbone: {backbone} ===")
    model = make_segformer_rgb(backbone, num_classes=1, pretrained=True).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    best_val_iou = -1.0
    best_path = OUTPUT_DIR / "models" / f"{backbone}_best_train_iou.pt"

    for epoch in range(1, EPOCHS+1):
        train_loss, train_metrics = run_one_epoch(model, train_loader, optimizer, use_amp=True)
        val_loss,   val_metrics   = run_one_epoch(model, val_loader, optimizer=None, use_amp=True)

        print(f"[{backbone}] Epoch {epoch}/{EPOCHS} — "
              f"train IoU={train_metrics['iou']:.4f}  val IoU={val_metrics['iou']:.4f}  "
              f"train Loss={train_loss:.4f}  val Loss={val_loss:.4f}")

        ts = datetime.utcnow().isoformat()
        pd.DataFrame([
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="train", loss=train_loss, **train_metrics),
            dict(timestamp=ts, backbone=backbone, epoch=epoch, split="val",   loss=val_loss,   **val_metrics),
        ]).to_csv(results_csv, mode="a", index=False, header=False)

        if val_metrics["iou"] > best_val_iou:
            best_val_iou = val_metrics["iou"]
            torch.save({
                "backbone": backbone,
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "best_val_iou": best_val_iou,
            }, best_path)

print(f"Training complete. Logs: {results_csv}")
print(f"Models saved under: {OUTPUT_DIR / 'models'}")



# 🧊 Mask2Former for Glacial Lakes (Semantic Segmentation)

This section adds a **Mask2Former** training pipeline (built on Detectron2) for **binary semantic segmentation** of glacial lakes (class `lake` vs `background`).  
Everything above remains unchanged; you can run your previous pipeline as-is, then run the cells below to train/evaluate Mask2Former on the **same image/mask folders**.


In [ ]:

# --- Install (run in your own environment; internet required) ---
# Choose correct torch/cu versions for your machine before installing detectron2.
# Example for CUDA 12.1 users (PyTorch >= 2.3):
# pip install --upgrade pip
# pip install 'git+https://github.com/facebookresearch/detectron2.git'
# pip install 'git+https://github.com/facebookresearch/Mask2Former.git'
#
# If you have trouble with builds, see:
# https://detectron2.readthedocs.io/en/latest/tutorials/install.html
# https://github.com/facebookresearch/Mask2Former
#
# NOTE: Do NOT run in this ChatGPT environment (no internet); run locally/Colab/kaggle.


In [ ]:

# --- Imports ---
import os, glob, random
from pathlib import Path
import numpy as np
import cv2
import torch

# Detectron2 / Mask2Former
from detectron2.config import get_cfg
from detectron2.engine import DefaultTrainer, default_setup
from detectron2.data import DatasetCatalog, MetadataCatalog, build_detection_test_loader
from detectron2.evaluation import SemSegEvaluator, inference_on_dataset
from detectron2.utils.visualizer import Visualizer, ColorMode
from detectron2.utils.logger import setup_logger

# Mask2Former config hook
from mask2former import add_maskformer2_config

setup_logger()
print("Torch:", torch.__version__)


In [ ]:

# --- Resolve dataset paths ---
# We try to reuse variables you may already have defined earlier in the notebook.
# If not found, fall back to placeholders: edit them to point to your data.
globals_dict = globals()

def _find_any(names):
    for n in names:
        if n in globals_dict and globals_dict[n]:
            return globals_dict[n]
    return None

# Candidate variable names coming from your earlier pipeline
train_img = _find_any(["TRAIN_IMAGES_DIR", "TRAIN_IMG_DIR", "train_images_dir", "train_img_dir", "TRAIN_DIR_IMAGES"])
train_msk = _find_any(["TRAIN_MASKS_DIR", "TRAIN_MASK_DIR", "train_masks_dir", "train_mask_dir", "TRAIN_DIR_MASKS"])
val_img   = _find_any(["VAL_IMAGES_DIR", "VAL_IMG_DIR", "val_images_dir", "val_img_dir", "VAL_DIR_IMAGES"])
val_msk   = _find_any(["VAL_MASKS_DIR", "VAL_MASK_DIR", "val_masks_dir", "val_mask_dir", "VAL_DIR_MASKS"])

# Placeholders (edit these if autodetect fails)
if train_img is None: train_img = "/path/to/train/images"
if train_msk is None: train_msk = "/path/to/train/masks"
if val_img   is None: val_img   = "/path/to/val/images"
if val_msk   is None: val_msk   = "/path/to/val/masks"

print("Train images:", train_img)
print("Train masks :", train_msk)
print("Val images  :", val_img)
print("Val masks   :", val_msk)

# Expected format:
# - Images: JPEG/PNG/TIF (Mask2Former works with common formats; prefer PNG/JPG for speed)
# - Masks: single-channel PNGs with values {0,1} (0=background, 1=lake)
#          If your masks are 0/255, they will be converted by the mapper below.


In [ ]:

# --- Dataset registration (semantic segmentation) ---
from detectron2.data import MetadataCatalog, DatasetCatalog

def _get_semseg_files(img_dir, msk_dir):
    img_paths = sorted([p for p in glob.glob(os.path.join(img_dir, "*")) if os.path.isfile(p)])
    samples = []
    for p in img_paths:
        base = os.path.splitext(os.path.basename(p))[0]
        # Try common mask extensions
        for ext in [".png", ".tif", ".tiff", ".jpg", ".jpeg"]:
            cand = os.path.join(msk_dir, base + ext)
            if os.path.exists(cand):
                samples.append((p, cand))
                break
    return samples

def register_semantic_folder(name, img_dir, msk_dir, stuff_classes=("background","lake")):
    DatasetCatalog.register(name, lambda: [
        {"file_name": ip, "sem_seg_file_name": mp} for ip, mp in _get_semseg_files(img_dir, msk_dir)
    ])
    MetadataCatalog.get(name).set(
        stuff_classes=list(stuff_classes),
        evaluator_type="sem_seg",
        ignore_label=255,   # will use 255 as ignore if needed
        sem_seg_root=msk_dir,
        image_root=img_dir,
        thing_classes=[],
        stuff_colors=[(0,0,0),(0,255,255)]
    )

register_semantic_folder("glacial_train", train_img, train_msk)
register_semantic_folder("glacial_val",   val_img,   val_msk)

print("Registered datasets:", list(DatasetCatalog.list()))
print("Classes:", MetadataCatalog.get("glacial_train").stuff_classes)


In [ ]:

# --- Mapper to ensure semantic mask is {0,1} ---
from detectron2.data import detection_utils as utils
from detectron2.data import transforms as T

class BinarySemSegMapper:
    def __init__(self, is_train=True, augment=True):
        self.is_train = is_train
        self.augment = augment

        # Geometric + photometric augs (tweak as needed)
        self.tfm_gens = []
        if augment and is_train:
            self.tfm_gens = [
                T.RandomFlip(prob=0.5, horizontal=True, vertical=False),
                T.RandomFlip(prob=0.2, horizontal=False, vertical=True),
                T.ResizeShortestEdge(short_edge_length=(512, 768), max_size=1024, sample_style="choice"),
            ]
        else:
            self.tfm_gens = [T.ResizeShortestEdge(short_edge_length=768, max_size=1280)]

    def __call__(self, dataset_dict):
        dataset_dict = dataset_dict.copy()
        image = utils.read_image(dataset_dict["file_name"], format="BGR")
        sem_seg = utils.read_image(dataset_dict["sem_seg_file_name"], "L")

        # Normalize mask to {0,1}
        # If mask uses 255 for lakes, convert to 1.
        # Otherwise, any non-zero becomes 1.
        sem_seg = (sem_seg > 0).astype("uint8")
        aug_input = T.AugInput(image, sem_seg=sem_seg)
        transforms = T.AugmentationList(self.tfm_gens)(aug_input)
        image, sem_seg = aug_input.image, aug_input.sem_seg

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())  # CHW uint8
        dataset_dict["sem_seg"] = torch.as_tensor(sem_seg.copy().astype("long"))  # HxW long
        return dataset_dict


In [ ]:

# --- Config ---
from detectron2.engine import DefaultTrainer
from detectron2.data import build_detection_train_loader

class Mask2FormerSemSegTrainer(DefaultTrainer):
    @classmethod
    def build_evaluator(cls, cfg, dataset_name, output_folder=None):
        if output_folder is None:
            output_folder = os.path.join(cfg.OUTPUT_DIR, "inference")
        return SemSegEvaluator(dataset_name, distributed=False, output_dir=output_folder)

    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=BinarySemSegMapper(is_train=True, augment=True))

    @classmethod
    def build_test_loader(cls, cfg, dataset_name):
        return build_detection_test_loader(cfg, dataset_name, mapper=BinarySemSegMapper(is_train=False, augment=False))

cfg = get_cfg()
add_maskformer2_config(cfg)

# Use a Mask2Former semantic segmentation config (ADE20K as base)
# You can choose a different backbone/config from the Mask2Former repo.
cfg.merge_from_file("configs/ade20k-150/semantic-segmentation/mask2former_swin_tiny_IN21k_384_bs16_160k.yaml")

# Datasets
cfg.DATASETS.TRAIN = ("glacial_train",)
cfg.DATASETS.TEST  = ("glacial_val",)
cfg.DATALOADER.NUM_WORKERS = 2

# Number of classes (background + lake)
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES = 2

# Pretrained weights (update to a valid path/URL if you have it)
# If you have local weights, set to that path. Otherwise rely on config's MODEL.WEIGHTS if available.
# Example (uncomment and set a real checkpoint path):
# cfg.MODEL.WEIGHTS = "/path/to/mask2former_swin_tiny_ade20k_pretrained.pth"

# Training schedule
cfg.SOLVER.IMS_PER_BATCH = 4
cfg.SOLVER.BASE_LR = 1e-4
cfg.SOLVER.MAX_ITER = 20000         # adjust based on dataset size
cfg.SOLVER.STEPS = []               # no LR decay
cfg.SOLVER.WARMUP_ITERS = 500

# Output
cfg.OUTPUT_DIR = "./output_mask2former_semseg"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Save the config
with open(os.path.join(cfg.OUTPUT_DIR, "config.yaml"), "w") as f:
    f.write(cfg.dump())

print("OUTPUT_DIR:", cfg.OUTPUT_DIR)
print("MAX_ITER  :", cfg.SOLVER.MAX_ITER)
print("LR        :", cfg.SOLVER.BASE_LR)


In [ ]:

# --- Train ---
trainer = Mask2FormerSemSegTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()


In [ ]:

# --- Evaluate ---
from detectron2.checkpoint import DetectionCheckpointer

cfg_eval = cfg.clone()
evaluator = SemSegEvaluator("glacial_val", distributed=False, output_dir=os.path.join(cfg.OUTPUT_DIR, "inference"))
val_loader = Mask2FormerSemSegTrainer.build_test_loader(cfg_eval, "glacial_val")

# Load best/latest model
checkpointer = DetectionCheckpointer(trainer.model)
# If you saved the best checkpoint path, load it here.
# Otherwise this uses the last iteration checkpoint in OUTPUT_DIR.
res = inference_on_dataset(trainer.model, val_loader, evaluator)
print("Validation metrics:", res)


In [ ]:

# --- Qualitative visualization ---
from detectron2.utils.visualizer import Visualizer
from PIL import Image
import matplotlib.pyplot as plt

metadata = MetadataCatalog.get("glacial_val")

def visualize_sample(img_path):
    im = cv2.imread(img_path)[:, :, ::-1]  # BGR->RGB
    with torch.no_grad():
        inputs = [{"image": torch.as_tensor(im.copy()).permute(2,0,1)}]
        output = trainer.model(inputs)[0]

    # For semantic seg, get per-pixel class map:
    sem = output["sem_seg"].argmax(dim=0).cpu().numpy().astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(im)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(sem, vmin=0, vmax=1)
    axes[1].set_title("Predicted mask (0=bg, 1=lake)")
    axes[1].axis("off")
    plt.show()

# visualize a few
sample_images = sorted(glob.glob(os.path.join(val_img, "*")))[:3]
for s in sample_images:
    visualize_sample(s)



### Notes & Tips
- **Masks must be single-channel with values {0,1}.** If your masks are 0/255, the mapper normalizes them to {0,1}.
- If training diverges (loss becomes NaN), try:
  - Lower learning rate (`1e-5` to `5e-5`).
  - Reduce `IMS_PER_BATCH` if you hit OOM.
  - Check masks for invalid pixels (NaN) or unexpected label values.
- You can switch backbones/configs by choosing another file from Mask2Former `configs/` and adjusting `NUM_CLASSES` and `MODEL.WEIGHTS` accordingly.
- For **class imbalance**, try focal loss configs or increase sampling of positive tiles when you prepare dataloaders (here, we rely on standard semantic setup).
